# Session B - experiments on an already-trained checkpoint

No training. This notebook attaches the output of `xtts-new-optimised` (run 5) and
spends its GPU time on the questions run 5 left open:

| | Question | Cost |
|---|---|---|
| 1 | What does fp16 **compute** cost? Run 5 never measured it - a dropped `--half` made that row a duplicate. | ~5 min |
| 2 | Is fp16 really indistinguishable from fp32, or was that one lucky seed? | 4 experiments |
| 3 | Does a lower temperature remove the last failures? | 2 experiments |
| 4 | Does one unified Sinhala->ASCII path at inference change anything? | 1 experiment |

**Why a separate kernel.** Pushing a new version of `xtts-new-optimised` would re-run
its 8.5 h training cell and replace its output - which holds the only surviving copy
of `best_model.pth`, `model_slim.pth` and `model_fp16.pth`. This one is read-only
towards all of that.

What run 5 settled and this does not revisit: the strip is lossless (963 tensors
bit-identical, metrics exactly equal), and the three size targets are met.

**`checkpoint_22000.pth` is gone** - it lived on `/kaggle/temp` and died with the
session, so the best-checkpoint question from run 5 cannot be reopened without
retraining. Only `best_model.pth` (step 14080) survived.

## 1. Install - restart the session after this cell

In [ ]:
# Same pins as the training notebook. coqui-tts is the maintained idiap fork; do
# NOT `pip install TTS`, which pins torch<2.1 and replaces Kaggle's CUDA build
# with a CPU wheel.
!pip install -q "coqui-tts>=0.25.1" "coqui-tts-trainer>=0.2.0" soundfile librosa tensorboard

## 1b. Preflight - can torch actually use this GPU?

`kernel-metadata.json` can ask for *a* GPU (`enable_gpu`) but not *which one* - the Kaggle
API has no accelerator-type field. Version 1 of this kernel was handed a **Tesla P100**
(sm_60), which the installed PyTorch does not build kernels for, and every CUDA call died
with `no kernel image is available for execution on the device`.

That failure surfaced 25 minutes in, at the first GPU call. This cell moves it to second
one, and says what to change.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("no GPU at all. Settings -> Accelerator -> GPU T4 x2.")

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = "sm_%d%d" % (major, minor)
built_for = torch.cuda.get_arch_list()
print("GPU            :", name)
print("capability     :", arch)
print("torch built for:", " ".join(built_for))

FIX = ("Fix: Settings -> Accelerator -> GPU T4 x2, then re-run. "
       "The Kaggle API has no accelerator-type field, so this cannot be set "
       "from kernel-metadata.json.")

# get_arch_list() is necessary but not sufficient -- the honest test is running a
# kernel. A P100 reports its capability quite happily and then fails on the first
# real op with "no kernel image is available for execution on the device".
try:
    _ = (torch.zeros(8, device="cuda") + 1).sum().item()
    torch.cuda.synchronize()
except Exception as exc:
    raise RuntimeError("this GPU (%s, %s) cannot run the installed PyTorch: %s | "
                       "torch has kernels for: %s | %s"
                       % (name, arch, exc, " ".join(built_for), FIX))

if arch not in built_for:
    raise RuntimeError("%s is %s but torch has kernels for %s. A tiny op passed, "
                       "but this will fail inside XTTS. %s"
                       % (name, arch, " ".join(built_for), FIX))

print("")
print("OK -- a CUDA kernel actually ran on this device.")

## 2. Code, inputs and paths

In [ ]:
import glob, os, shutil, subprocess

REPO = "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git"
# Cloned into /kaggle/temp, NOT /kaggle/working. Run 5 cloned into the output
# volume, so its whole .git tree ended up in the kernel output and drowned the
# result files in every listing and download.
CODE = "/kaggle/temp/code"
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", REPO, CODE], check=True)
SRC = CODE + "/xtts_model_female"
OPT = CODE + "/xtts_model_female_optimized"
print("code at", subprocess.run(["git", "-C", CODE, "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())


def sh(args, cwd=SRC):
    """Run a step and STOP the notebook if it fails."""
    print("$", " ".join(args), flush=True)
    r = subprocess.run(args, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError("step failed with exit %d: %s" % (r.returncode, " ".join(args)))


# ---- the attached run-5 output -------------------------------------------
# Located by a file we know it contains rather than by mount path, because
# Kaggle nests inputs differently depending on how they were attached.
hits = glob.glob("/kaggle/input/**/model_slim.pth", recursive=True)
if not hits:
    raise RuntimeError(
        "run 5's output is not attached. Add Input -> Notebook Output -> "
        "uom230429e/xtts-new-optimised, or set kernel_sources in "
        "kernel-metadata.json.")
EXPORT = os.path.dirname(min(hits, key=len))          # .../xtts_si_female
CK   = EXPORT + "/model.pth"                          # 5.6 GB trainer checkpoint
SLIM = EXPORT + "/model_slim.pth"                     # 1.87 GB, bit-identical
FP16 = EXPORT + "/model_fp16.pth"                     # 0.93 GB

runs = glob.glob("/kaggle/input/**/GPT_XTTS_si_female-*", recursive=True)
run = max(runs, key=len) if runs else None

# ---- the VoiceMakers corpus ----------------------------------------------
hits = [p for p in glob.glob("/kaggle/input/**/*", recursive=True)
        if os.path.isdir(p) and "dinithi" in os.path.basename(p).lower()]
if not hits:
    raise RuntimeError("VoiceMakers dataset not attached.")
DATA = os.path.dirname(min(hits, key=lambda p: len(p.split("/"))))

DATASET = "/kaggle/temp/female_dataset"
BASE    = "/kaggle/temp/base"
OUT     = "/kaggle/working/experiments"
RESULTS = OUT + "/results.csv"
os.makedirs(OUT, exist_ok=True)

for name, p in (("export", EXPORT), ("run", run), ("data", DATA)):
    print("%-8s: %s" % (name, p))
for name, p in (("ck", CK), ("slim", SLIM), ("fp16", FP16)):
    print("%-8s: %.3f GB" % (name, os.path.getsize(p) / 1e9)
          if os.path.isfile(p) else "%-8s: MISSING" % name)

## 3. Base files and dataset

`evaluate_xtts.py` reads only `config.json` and `vocab.json` from `--base`; the weights
come from `--checkpoint`. Both files are already in the attached export, so nothing is
downloaded. The dataset has to be rebuilt because `/kaggle/temp` does not survive a
session - same seed, so the eval split is identical to run 5's.

In [ ]:
import os, shutil

os.makedirs(BASE, exist_ok=True)
for f in ("config.json", "vocab.json"):
    if not os.path.isfile(BASE + "/" + f):
        shutil.copy2(EXPORT + "/" + f, BASE + "/" + f)
print("base:", os.listdir(BASE))

# --seed 1234 and --eval-per-speaker 40 match run 5, so eval_reference.json holds
# the same held-out clips. Change either and nothing here is comparable.
sh(["python", "prepare_voicemakers.py", "--src", DATA, "--out", DATASET,
    "--speakers", "dinithi", "harini", "--vocab", BASE + "/vocab.json",
    "--eval-per-speaker", "40", "--seed", "1234"])

## 4. Carry run 5's results forward

Copied into this session's register so the final table shows the new rows beside the
ones they have to be judged against. Experiment ids are stable, so nothing collides.

In [ ]:
import glob, shutil

prev = glob.glob("/kaggle/input/**/experiments/experiments.csv", recursive=True)
if prev:
    shutil.copy2(prev[0], OUT + "/experiments.csv")
    print("carried forward:", sum(1 for _ in open(prev[0])) - 1, "rows from run 5")
else:
    print("run 5's register not found -- new rows will stand alone")

## 5. Question 1 - what fp16 *compute* costs

Storage and arithmetic are different things. Run 5 measured the fp16 **file** and found
it changes nothing but size and load time: peak VRAM was 2.47 GB for every
configuration, because `load_state_dict` casts each tensor to the dtype of the
parameter receiving it, so an fp16 file becomes an fp32 model in VRAM.

`--half` is the other thing - arithmetic actually in half precision. It is the only
lever here that can move RTF, and it changes results, so it would need its own quality
gate before shipping. Measured, not shipped.

In [ ]:
import glob, json, os

refs = (run + "/speaker_refs.json") if run else None
if refs and os.path.isfile(refs):
    REF = sorted(json.load(open(refs)).values())[0]
    if not os.path.isfile(REF):          # recorded against run 5's /kaggle/temp
        REF = sorted(glob.glob(DATASET + "/wavs/*/*.wav"))[0]
else:
    REF = sorted(glob.glob(DATASET + "/wavs/*/*.wav"))[0]
print("reference wav:", REF)

# The row run 5 was supposed to produce. `extra` was built and never passed, so
# that row came back half=0 -- a duplicate of slim-fp32 wearing another name.
# NOT fatal. This is one optional row; the sweeps below are the point of the
# session. Two earlier attempts died here -- once on a P100, once on a dtype
# mismatch in --half -- and took three hours of experiments down with them
# before a single one had run.
try:
    sh(["python", "benchmark.py", "--checkpoint", SLIM, "--base", BASE,
        "--ref", REF, "--tag", "fp16-compute", "--out", RESULTS, "--half"], cwd=OPT)
except Exception as exc:
    print("fp16-compute benchmark FAILED, continuing to the sweeps:", exc)

if os.path.isfile(RESULTS):
    print(open(RESULTS, encoding="utf-8").read())

## 6. Questions 2-4 - the sweeps

Three separate calls, one question each, all appending to one register. A single
product of every list would be dozens of evaluations and would not say which knob
moved anything.

Roughly 20 minutes per experiment (40 clips per speaker, synthesis plus `pyin`, plus
UTMOS), so ~2.5 h for the seven below.

In [ ]:
# Q2 -- is fp16 really indistinguishable from fp32?
#
# Run 5 compared them at seed 1234 and fp16 came out nominally better on four of
# five metrics. That is not a result: every delta sat inside the run-to-run noise
# floor (MCD +/-0.30 dB, F0 corr +/-0.045, failure rate +/-2.5 pt). Two more seeds
# say whether anything survives resampling. Seed 1234 is already measured and is
# carried forward, so only 1235 and 1236 are run here.
sh(["python", "sweep_eval.py", "--run", run or DATASET, "--base", BASE,
    "--dataset", DATASET, "--out", OUT, "--registry", OUT + "/experiments.csv",
    "--checkpoints", SLIM, FP16, "--seeds", "1235,1236", "--n", "40", "--utmos"])

In [ ]:
# Q3 -- can a lower temperature remove the last failures?
#
# The deployment candidate already reports 0.0 % failures and duration ratio 1.005 at
# t=0.75, so there is less headroom here than run 4 suggested. Worth two experiments
# to see whether 0.65/0.70 holds that while tightening duration, not worth ten.
sh(["python", "sweep_eval.py", "--run", run or DATASET, "--base", BASE,
    "--dataset", DATASET, "--out", OUT, "--registry", OUT + "/experiments.csv",
    "--checkpoints", FP16, "--temperature", "0.65,0.7", "--n", "40", "--utmos"])

In [ ]:
# Q4 -- one text path for both speakers, at inference.
#
# dinithi's training text came from fold(romanisation), harini's from
# sinhala_to_ascii(script), because her metadata has no romanised column. Those
# agree on ~96.6 % of lines. --text-from script re-derives every clip's text
# through the transliterator, so both speakers go down one path.
#
# This is the cheap HALF of the experiment: it changes what is spoken, not what the
# model was trained on. A real answer needs prepare_voicemakers.py --text-path script
# and a retrain. If this moves harini's numbers, that retrain is worth 8.5 h.
sh(["python", "sweep_eval.py", "--run", run or DATASET, "--base", BASE,
    "--dataset", DATASET, "--out", OUT, "--registry", OUT + "/experiments.csv",
    "--checkpoints", FP16, "--text-from", "script", "--n", "40", "--utmos"])

## 7. The register

In [ ]:
from IPython.display import Markdown, display
import os

for f in sorted(os.listdir(OUT)):
    if f.endswith("_summary.md"):
        display(Markdown(open(OUT + "/" + f, encoding="utf-8").read()))

print(open(OUT + "/experiments.csv", encoding="utf-8").read())
if os.path.isfile(RESULTS):
    print(open(RESULTS, encoding="utf-8").read())

# Ordering in those tables is not a verdict. Run 5's noise floor -- MCD +/-0.30 dB,
# F0 corr +/-0.045, failure rate +/-2.5 pt, UTMOS +/-0.01 -- is what decides whether
# two rows differ, and 2.5 % of 80 clips is two clips.
print("Download /kaggle/working/experiments before the session ends.")